# Hidden Poetic Schools of Pre-Islamic Poetry

One notebook. Everything inside. Run each cell top to bottom with **Shift+Enter**.

In [ ]:
# CELL 1 — Install packages
!pip install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn requests beautifulsoup4 lxml tqdm scipy

In [ ]:
# CELL 2 — Settings
import random, re, sqlite3, time, numpy as np, pandas as pd
from pathlib import Path
from urllib.parse import urljoin

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DB_PATH = Path("poems.db")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

SBERT_MODEL = "akhooli/Arabic-SBERT-100K"
MIN_POEMS  = 2
MIN_VERSES = 2

UMAP_NEIGHBORS = 15
UMAP_DIST      = 0.1
UMAP_DIMS      = 10
HDBSCAN_MIN    = 5

BASE_URL     = "https://www.aldiwan.net"
CATEGORY_URL = BASE_URL + "/cat-poets-pre-islamic-period"
DELAY = 1.5

print("Settings loaded.")

In [ ]:
# CELL 3 — Scrape poems from aldiwan.net
# First run: uses limit=5 to test. Once it works, change limit=None and re-run.
import requests as req
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

session = req.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

def fetch(url):
    r = session.get(url, timeout=30)
    r.raise_for_status()
    time.sleep(DELAY)
    return BeautifulSoup(r.text, "lxml")

def is_arabic(text):
    if not text or len(text.strip()) < 3:
        return False
    return sum(1 for c in text if '\u0600' <= c <= '\u06FF') >= 3

# Create DB
conn = sqlite3.connect(DB_PATH)
conn.executescript("""
    CREATE TABLE IF NOT EXISTS poets  (poet_id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, slug TEXT UNIQUE, bio TEXT);
    CREATE TABLE IF NOT EXISTS poems  (poem_id INTEGER PRIMARY KEY AUTOINCREMENT, poet_id INTEGER, title TEXT, url TEXT UNIQUE, n_verses INTEGER);
    CREATE TABLE IF NOT EXISTS verses (verse_id INTEGER PRIMARY KEY AUTOINCREMENT, poem_id INTEGER, verse_order INTEGER, text TEXT);
""")
conn.commit()

# Get poet list
soup = fetch(CATEGORY_URL)
poet_links = []
seen = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    if "/cat-poet-" not in href:
        continue
    url = urljoin(BASE_URL, href)
    name = a.get_text(strip=True)
    if name and url not in seen:
        seen.add(url)
        poet_links.append({"name": name, "url": url})

print(f"Found {len(poet_links)} poets.")

# === CHANGE THIS TO None FOR FULL SCRAPE ===
limit = 5
# ===========================================

if limit:
    poet_links = poet_links[:limit]

for poet in tqdm(poet_links, desc="Poets"):
    slug = poet["url"].rstrip("/").split("/")[-1]
    if conn.execute("SELECT 1 FROM poets WHERE slug=?", (slug,)).fetchone():
        continue
    try:
        ps = fetch(poet["url"])
    except Exception as e:
        print(f"Skip {poet['name']}: {e}")
        continue

    bio_tag = ps.find("h4")
    bio = bio_tag.get_text(strip=True) if bio_tag else ""
    conn.execute("INSERT OR IGNORE INTO poets (name,slug,bio) VALUES (?,?,?)", (poet["name"], slug, bio))
    conn.commit()
    pid = conn.execute("SELECT poet_id FROM poets WHERE slug=?", (slug,)).fetchone()[0]

    poem_links = []
    pseen = set()
    for a in ps.find_all("a", href=True):
        href = a["href"]
        m = re.search(r"/poem(\d+)\.html", href)
        if not m:
            continue
        purl = urljoin(BASE_URL, href)
        title = a.get_text(strip=True)
        if title and purl not in pseen:
            pseen.add(purl)
            poem_links.append({"title": title, "url": purl})

    for poem in poem_links:
        if conn.execute("SELECT 1 FROM poems WHERE url=?", (poem["url"],)).fetchone():
            continue
        try:
            pmsoup = fetch(poem["url"])
        except:
            continue
        verses = [t.get_text(strip=True) for t in pmsoup.select("h3")]
        verses = [v for v in verses if is_arabic(v)]
        if len(verses) < MIN_VERSES:
            continue
        cur = conn.execute("INSERT INTO poems (poet_id,title,url,n_verses) VALUES (?,?,?,?)",
                           (pid, poem["title"], poem["url"], len(verses)))
        pmid = cur.lastrowid
        conn.executemany("INSERT INTO verses (poem_id,verse_order,text) VALUES (?,?,?)",
                         [(pmid, i, v) for i, v in enumerate(verses)])
        conn.commit()

np_ = conn.execute("SELECT COUNT(*) FROM poets").fetchone()[0]
npm = conn.execute("SELECT COUNT(*) FROM poems").fetchone()[0]
nv  = conn.execute("SELECT COUNT(*) FROM verses").fetchone()[0]
conn.close()
print(f"Done! poets={np_}  poems={npm}  verses={nv}")

**Did you see poets/poems/verses above?** If yes, great — go back up, change `limit = 5` to `limit = None`, and re-run cell 3 for the full scrape (takes a couple hours, safe to re-run).

In [ ]:
# CELL 4 — Load and clean the data
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("""
    SELECT p.name AS poet, pm.poem_id, v.verse_order, v.text
    FROM verses v
    JOIN poems pm ON v.poem_id = pm.poem_id
    JOIN poets p  ON pm.poet_id = p.poet_id
    ORDER BY p.poet_id, pm.poem_id, v.verse_order
""", conn)
conn.close()

def clean(text):
    # strip diacritics
    text = re.sub("[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0670]", "", text)
    text = re.sub("\u0640", "", text)                  # tatweel
    text = re.sub("[\u0625\u0623\u0622\u0627]", "\u0627", text)  # alef variants
    text = re.sub("\u0649", "\u064A", text)              # ya
    text = re.sub("\u0629", "\u0647", text)              # ta marbuta
    return re.sub(r"\s+", " ", text).strip()

df["text"] = df["text"].apply(clean)
df = df[df["text"].str.len() > 2]
df = df[df.groupby("poet")["poem_id"].transform("nunique") >= MIN_POEMS]

poems_by_poet = {}
for poet, g in df.groupby("poet"):
    poems_by_poet[poet] = [sub.sort_values("verse_order")["text"].tolist()
                           for _, sub in g.groupby("poem_id")]

print(f"Poets: {len(poems_by_poet)}  |  Poems: {df['poem_id'].nunique()}  |  Verses: {len(df)}")

In [ ]:
# CELL 5 — Embed every poet with Arabic Sentence-BERT
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(SBERT_MODEL)

poet_embeddings = {}
for poet, poems in tqdm(poems_by_poet.items(), desc="Embedding"):
    poem_vecs = []
    for verses in poems:
        if not verses:
            continue
        embs = model.encode(verses, show_progress_bar=False, normalize_embeddings=True)
        poem_vecs.append(embs.mean(axis=0))
    if poem_vecs:
        poet_embeddings[poet] = np.mean(poem_vecs, axis=0)

print(f"Embedded {len(poet_embeddings)} poets.")

In [ ]:
# CELL 6 — Cluster the poets into schools
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

poet_names = sorted(poet_embeddings.keys())
emb_matrix = np.stack([poet_embeddings[n] for n in poet_names])
sim_matrix = cosine_similarity(emb_matrix)

# High-dim reduction for clustering
reduced = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_DIST,
                     n_components=UMAP_DIMS, metric="cosine",
                     random_state=RANDOM_SEED).fit_transform(emb_matrix)

labels = hdbscan.HDBSCAN(min_cluster_size=HDBSCAN_MIN).fit_predict(reduced)

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
mask = labels != -1
sil = silhouette_score(reduced[mask], labels[mask]) if n_clusters >= 2 and mask.sum() > n_clusters else None

# 2D reduction for the picture
coords = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_DIST,
                    n_components=2, metric="cosine",
                    random_state=RANDOM_SEED).fit_transform(emb_matrix)

print(f"Clusters: {n_clusters}   Silhouette: {sil}")

In [ ]:
# CELL 7 — Show the picture
import matplotlib.pyplot as plt
import seaborn as sns

unique = sorted(set(labels))
pal = sns.color_palette("husl", len([l for l in unique if l != -1]))
colors = {}
ci = 0
for l in unique:
    if l == -1:
        colors[l] = (0.7, 0.7, 0.7)
    else:
        colors[l] = pal[ci]; ci += 1

plt.figure(figsize=(10, 8))
for l in unique:
    sel = labels == l
    plt.scatter(coords[sel, 0], coords[sel, 1], s=40, alpha=0.85,
                color=colors[l], label="Noise" if l == -1 else f"School {l}",
                edgecolor="k", linewidth=0.3)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Hidden Stylistic Schools of Pre-Islamic Poetry")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "clusters.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to figures/clusters.png")

In [ ]:
# CELL 8 — Save results
result = pd.DataFrame({"poet": poet_names, "cluster": labels}).sort_values("cluster")
result.to_csv("cluster_assignments.csv", index=False)
print("Saved to cluster_assignments.csv")
result

## Done!\n\n- `poems.db` — scraped corpus\n- `figures/clusters.png` — the schools picture\n- `cluster_assignments.csv` — which poet is in which cluster